In [1]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Subset Olink RVAT significant

# olink_burden_test_file = "proteomics_prs_am_loftee_mac20_burden_regression_results.parquet"
# olink_burden_test_file = "proteomics_prs_loftee_mac20_burden_regression_results.parquet"
olink_burden_test_file = "proteomics_prs_df_loftee_mac20_burden_regression_results.parquet"
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/blacklist/{olink_burden_test_file} -o /home/dnanexus/data_dir/

olink_whitelist = (
    pl.read_parquet(f'/home/dnanexus/data_dir/{olink_burden_test_file}')
    .rename({'gene': 'region'})
    .filter((pl.col('padj')<=0.05) & (pl.col('wilcox_padj')<=0.05) )
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

# olink_correlations_file = "olink_all_mac20_lofteeHC_correlations.parquet"
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/{olink_correlations_file} -o /home/dnanexus/data_dir/

# olink_corrs = (
#     pl.read_parquet(f'/home/dnanexus/data_dir/{olink_correlations_file}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .drop_nans()
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# olink_whitelist = (
#     olink_whitelist
#     .join(olink_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     # .sort('loftee_corr_abs', descending=True)
#     # .sort('pval_fdr')
#     # .unique(subset=["region"], keep="first", maintain_order=True)
# )

olink_whitelist

[===========================================================>] Completed 176,986 of 176,986 bytes (100%) /home/dnanexus/data_dir/proteomics_prs_df_loftee_mac20_burden_regression_results.parquett


region,phenotype
str,str
"""ENSG00000116675""","""ENSG00000116675_olink"""
"""ENSG00000123609""","""ENSG00000123609_olink"""
"""ENSG00000167851""","""ENSG00000167851_olink"""
"""ENSG00000156711""","""ENSG00000156711_olink"""
"""ENSG00000174990""","""ENSG00000174990_olink"""
…,…
"""ENSG00000136542""","""ENSG00000136542_olink"""
"""ENSG00000196576""","""ENSG00000196576_olink"""
"""ENSG00000111144""","""ENSG00000111144_olink"""


In [3]:
# EUR unrelated individuals

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/sample_lists/olink_cauc_3rd_degree_samples.csv -o /home/dnanexus/data_dir/

unrel_eur_samples = pl.read_csv('/home/dnanexus/data_dir/olink_cauc_3rd_degree_samples.csv')['sample'].cast(pl.Utf8).to_list()
print(len(unrel_eur_samples))
unrel_eur_samples[:5]

[===========================================================>] Completed 324,999 of 324,999 bytes (100%) /home/dnanexus/data_dir/olink_cauc_3rd_degree_samples.csvv
40624


['5645319', '5959139', '5673208', '5732867', '2074480']

In [4]:
# prot_file = "cauc_cov_regression_90pcs_prs"  # covariates and PRS corrected
# prot_file = "cauc_protrider_lite_prs_rint"   # PROTRIDER corrected (RINT)
prot_file = "cauc_protrider_lite_prs_t_df"     # PROTRIDER corrected (T distribution)

# Download Olink:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/{prot_file}.parquet -o /home/dnanexus/data_dir/

phenos = pl.read_parquet(f'/home/dnanexus/data_dir/{prot_file}.parquet')

olink_genes2use = list(set(phenos.columns).intersection(set(olink_whitelist['region'])))

phenos = phenos.select(['sample'] + olink_genes2use)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes2use,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        # region = pl.col('phenotype'),
        phenotype = pl.col('phenotype') + '_olink',
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

[===========================================================>] Completed 1,034,847,804 of 1,034,847,804 bytes (100%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df.parquett
shape: (1_153, 2)
┌───────────────────────┬───────┐
│ phenotype             ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u64   │
╞═══════════════════════╪═══════╡
│ ENSG00000066056_olink ┆ 39208 │
│ ENSG00000146648_olink ┆ 39208 │
│ ENSG00000179776_olink ┆ 39208 │
│ ENSG00000163359_olink ┆ 39208 │
│ ENSG00000076662_olink ┆ 39208 │
│ …                     ┆ …     │
│ ENSG00000125730_olink ┆ 31897 │
│ ENSG00000184530_olink ┆ 31679 │
│ ENSG00000000971_olink ┆ 31668 │
│ ENSG00000152229_olink ┆ 31635 │
│ ENSG00000111405_olink ┆ 30953 │
└───────────────────────┴───────┘


sample,phenotype,pheno_value
str,str,f64
"""5645319""","""ENSG00000141642_olink""",-2.479645
"""5959139""","""ENSG00000141642_olink""",-0.884029
"""5732867""","""ENSG00000141642_olink""",-1.787154
"""2074480""","""ENSG00000141642_olink""",-0.846427
"""4659532""","""ENSG00000141642_olink""",-0.281791
…,…,…
"""1807196""","""ENSG00000171246_olink""",-1.842065
"""4223555""","""ENSG00000171246_olink""",-0.394956
"""2992773""","""ENSG00000171246_olink""",-0.671446


In [6]:
long_phenos.filter(pl.col('phenotype')=='ENSG00000141642_olink').select(pl.col('pheno_value').std())

pheno_value
f64
0.999016


In [3]:
mac = 20

RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

LOCAL_DIR = '/home/dnanexus/data_dir'

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(olink_whitelist.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

Error: path
"/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


/tmp/ipykernel_530332/966521399.py:20: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


id
str
"""chr5:83967060:C:G"""
"""chr14:94439962:C:T"""
"""chr2:102385547:A:T"""
"""chr12:44853831:A:T"""
"""chr7:154032319:G:C"""
…
"""chr1:21222278:C:T"""
"""chr3:15615300:C:T"""
"""chr5:78073941:T:C"""


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(
        pl.col('gt')==1,
        pl.col('sample').is_in(unrel_eur_samples),
    )
    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

Error: path "/home/dnanexus/data_dir/gt_long.parquet" already exists but
-f/--overwrite was not set


id,sample,gt
str,str,i8
"""chr10:20020015:CT:C""","""1235710""",1
"""chr10:20020015:CT:C""","""4775888""",1
"""chr10:20020019:T:C""","""2175143""",1
"""chr10:20020023:T:C""","""3380428""",1
"""chr10:20020033:G:T""","""4917981""",1
…,…,…
"""chr1:173491128:C:T""","""2282781""",1
"""chr1:173491128:C:T""","""1757946""",1
"""chr1:173491128:C:T""","""3934103""",1


In [7]:
import math

output_dir = '/home/dnanexus/data_dir/appv_olink'
!mkdir -p {output_dir}

pheno_list = long_phenos['phenotype'].unique().to_list()
CHUNK_SIZE = 100
num_phenos = len(pheno_list)
num_chunks = math.ceil(num_phenos / CHUNK_SIZE)

# Process in Batches
for i in tqdm(range(0, num_phenos, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_phenos = pheno_list[i : i + CHUNK_SIZE]

    print(f"Processing chunk starting at index: {i}")
    (
        long_phenos.lazy()
        .filter(pl.col('phenotype').is_in(chunk_phenos))
        .join(
            long_gt.lazy(),
            on='sample',
            how='inner'
        )
        .group_by(['id', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(descending=True, method="max")      # Olink is always under expression
                .over('phenotype')
                .cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len().over('phenotype')
            ).cast(pl.Float32),
        )

        .sink_parquet(f'{output_dir}/tmp_appv_chunk_{i}.parquet')
    )


  0%|          | 0/12 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  8%|▊         | 1/12 [01:27<16:06, 87.89s/it]

Processing chunk starting at index: 100


 17%|█▋        | 2/12 [02:52<14:16, 85.70s/it]

...

 83%|████████▎ | 10/12 [14:19<02:52, 86.12s/it]

Processing chunk starting at index: 1000


 92%|█████████▏| 11/12 [15:45<01:26, 86.02s/it]

Processing chunk starting at index: 1100


100%|██████████| 12/12 [16:33<00:00, 82.76s/it]


In [4]:
# Create small appv file directly from chunks — no 92GB intermediate
# Each raw chunk (~8GB) gets filtered to gene-trait associations (~50MB), then combined.
output_dir = '/home/dnanexus/data_dir/appv_olink'
small_output_file = "/home/dnanexus/data_dir/olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet"

# Pre-collect id_region filtered to whitelist regions only (much smaller than full anno)
id_region = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(pl.col('region').is_in(olink_whitelist['region']))
    .select(['id', 'region'])
    .unique()
    .collect()
)
print(f"id_region: {id_region.shape}")

# Filter each raw chunk → gene-trait associations, deduplicate, collect
raw_files = sorted(glob.glob(f'{output_dir}/tmp_appv_chunk_*.parquet'))
filtered_chunks = []

for f in tqdm(raw_files):
    chunk = (
        pl.scan_parquet(f)
        .join(id_region.lazy(), on='id', how='inner')
        .join(olink_whitelist.lazy(), on=['region', 'phenotype'], how='inner')
        .drop('region')
        .unique(subset=['id', 'phenotype'])
        .collect(engine='streaming')
    )
    filtered_chunks.append(chunk)
    print(f"  {f.split('/')[-1]}: {chunk.shape}")

result = pl.concat(filtered_chunks)
print(f"\nCombined: {result.shape}")
result.write_parquet(small_output_file)
print(f"Written to {small_output_file}")

/tmp/ipykernel_530332/1165544854.py:12: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


id_region: (23137583, 2)


  8%|▊         | 1/12 [00:16<03:01, 16.52s/it]

  tmp_appv_chunk_0.parquet: (364656, 7)


 17%|█▋        | 2/12 [00:33<02:45, 16.51s/it]

...

 92%|█████████▏| 11/12 [02:42<00:14, 14.76s/it]

  tmp_appv_chunk_800.parquet: (696199, 7)


100%|██████████| 12/12 [02:56<00:00, 14.74s/it]

  tmp_appv_chunk_900.parquet: (465648, 7)

Combined: (5084046, 7)


Written to /home/dnanexus/data_dir/olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet


In [5]:
# Verify small file
pl.scan_parquet(small_output_file).head().collect()

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,i32,f32,f32,f32,f32
"""chr12:24805471:G:A""","""ENSG00000060982_olink""",1,0.328067,null,1.720632e6,0.36397
"""chr12:122768837:C:G""","""ENSG00000139726_olink""",1,-1.312343,null,4.311464e6,0.933468
"""chr2:102332039:T:C""","""ENSG00000115604_olink""",1,0.249114,null,2.082593e6,0.392066
"""chr2:37346399:G:A""","""ENSG00000115828_olink""",1,1.00197,null,705046.0,0.133705
"""chr16:89195863:C:G""","""ENSG00000129910_olink""",1,-0.059154,null,2.813642e6,0.530968


In [6]:
!dx upload {small_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 100,324,032 of 100,324,032 bytes (100%) /home/dnanexus/data_dir/olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_small_desc.parquet
ID                                file-J66Pk8jJg0yJJ0bQKzkB9qvg
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/ukbgym/avg_pheno_per_var
Name                              olink_protrider_t_df_loftee_mac20_EURunrelated_appv_percentiles_sm
                                  all_desc.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Wed Feb 11 22:03:15 2026
Created by                        shubhankar
 via the job                      job-J669QQ0Jg0yFfB